In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

model_name = "bert-base-uncased" #biobert, chemberta v2 

tokenizer = AutoTokenizer.from_pretrained(model_name, do_lower_case=True)
model = AutoModel.from_pretrained(model_name, output_hidden_states=True)

model.eval()

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 4485.36it/s]
BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False)
  

In [22]:
def get_embedding(smiles):
    inputs = tokenizer(
        smiles,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=128
    )
    
    with torch.no_grad():
        outputs = model(**inputs)
    
    # CLS token embedding (standard choice)
    embedding = outputs.last_hidden_state[:, 0, :]
    
    return embedding.squeeze(0)

In [23]:
def batch_embeddings(smiles_list, batch_size=32):
    all_embeddings = []
    
    for i in range(0, len(smiles_list), batch_size):
        batch = smiles_list[i:i+batch_size]
        
        inputs = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=128
        )
        
        with torch.no_grad():
            outputs = model(**inputs)
        
        batch_emb = outputs.last_hidden_state[:, 0, :]
        all_embeddings.append(batch_emb)
    
    return torch.cat(all_embeddings, dim=0)

In [24]:
import pandas as pd

df = pd.read_csv("C:\\Users\\Diya Budlakoti\\Desktop\\SIN_proj\\dataset\\final_processed_data.csv")
unique_smiles = list(set(df["SMILES1"]) | set(df["SMILES2"]))

unique_embeddings = batch_embeddings(unique_smiles)

# Create mapping
smiles_to_emb = dict(zip(unique_smiles, unique_embeddings))
emb_s1 = torch.stack([smiles_to_emb[s] for s in df["SMILES1"]])
emb_s2 = torch.stack([smiles_to_emb[s] for s in df["SMILES2"]])

In [25]:
combined_embeddings = []

for e1, e2 in zip(emb_s1, emb_s2):
    combined_embeddings.append(e1 + e2)

combined_embeddings = torch.stack(combined_embeddings)

In [26]:
embedding_cache = {}
def get_embedding_cached(smiles):
    if smiles in embedding_cache:
        return embedding_cache[smiles]
    
    emb = get_embedding(smiles)
    embedding_cache[smiles] = emb
    return emb

In [27]:
for s in df["SMILES1"]:
    get_embedding_cached(s)

for s in df["SMILES2"]:
    get_embedding_cached(s)
print(len(embedding_cache))

644


In [28]:
torch.save(embedding_cache, "smiles_embeddings.pt")